# temporal_event_model v1 real architecture inspection

This notebook builds the real frozen masked-event encoder from a v20 checkpoint, freezes it, attaches the temporal v1 head, and then shows dummy inference shapes, `torchinfo` summary, and a `torchview` diagram when those packages are available.

The default render context is intentionally small (`CONTEXT_CHUNKS = 4`) so diagram/summary tracing is practical on a laptop. Set it to `64` when you want the exact temporal training context shape.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'research').exists()), Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
import torch
from torch import nn

from research.masked_event_model.v20.config import ModelConfig as V20ModelConfig
from research.masked_event_model.v20.model import EventTokenMaskedAutoencoder
from research.temporal_event_model.v1.config import ModelConfig as TemporalModelConfig
from research.temporal_event_model.v1.model import TemporalEventPredictor

WORKSTATION_ROOT = Path(r"\\DESKTOP-SAAI85T\Workstation-D")
V20_RUN = WORKSTATION_ROOT / "TradingML" / "runtimes" / "masked_event_model" / "v20" / "pretrain" / "v20-fullpretrain-sharddecay-fixedmask070-emb32-bs8192-3epochs"
CHECKPOINTS = {
    "epoch1": V20_RUN / "checkpoints" / "checkpoint_step_000130176.pt",
    "epoch2": V20_RUN / "checkpoints" / "checkpoint_step_000260352.pt",
    "latest": V20_RUN / "checkpoints" / "checkpoint_latest.pt",
}

# Choose the pretrained event encoder checkpoint here.
ENCODER_CHECKPOINT_NAME = "epoch2"
ENCODER_CHECKPOINT = CHECKPOINTS[ENCODER_CHECKPOINT_NAME]

# Optional: set this to a trained temporal-v1 checkpoint if/when one exists.
# Leave empty to inspect the real architecture with a randomly initialized temporal head.
TEMPORAL_CHECKPOINT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 1
CONTEXT_CHUNKS = 4
TARGET_CHUNKS = 1
EVENTS_PER_CHUNK = 128
HEADER_BYTES = 14
EVENT_BYTES = 16
BITS_PER_BYTE = 8

print(f"device={DEVICE}")
print(f"encoder_checkpoint={ENCODER_CHECKPOINT}")
print(f"temporal_checkpoint={TEMPORAL_CHECKPOINT if TEMPORAL_CHECKPOINT else '<none>'}")
print(f"render_input: batch={BATCH_SIZE}, context_chunks={CONTEXT_CHUNKS}, target_chunks={TARGET_CHUNKS}")

In [ ]:
def load_v20_encoder(checkpoint: Path, device: torch.device) -> nn.Module:
    if not checkpoint.exists():
        raise FileNotFoundError(f"Encoder checkpoint not found: {checkpoint}")
    encoder_config = V20ModelConfig(
        input_representation="bit",
        d_byte=40,
        d_model=256,
        embedding_dim=32,
        n_heads=8,
        encoder_layers=10,
        decoder_layers=4,
        ffn_mult=4,
        dropout=0.08,
        decoder_force_fp32=False,
    )
    autoencoder = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=encoder_config)
    payload = torch.load(checkpoint, map_location="cpu")
    if isinstance(payload, dict):
        state = payload.get("model_state_dict") or payload.get("model") or payload.get("state_dict")
    else:
        state = payload
    if not isinstance(state, dict):
        raise RuntimeError(f"Checkpoint does not contain a model state dict: {checkpoint}")
    missing, unexpected = autoencoder.load_state_dict(state, strict=False)
    if missing:
        print(f"encoder load missing keys: {len(missing)}")
    if unexpected:
        print(f"encoder load unexpected keys: {len(unexpected)}")
    encoder = autoencoder.build_encoder_model().to(device).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    return encoder


def build_temporal_model(device: torch.device) -> TemporalEventPredictor:
    event_encoder = load_v20_encoder(ENCODER_CHECKPOINT, device)
    temporal_config = TemporalModelConfig(
        embedding_dim=32,
        temporal_d_model=256,
        temporal_layers=6,
        temporal_heads=8,
        temporal_ffn_mult=4,
        decoder_layers=2,
        dropout=0.08,
    )
    model = TemporalEventPredictor(
        event_encoder=event_encoder,
        config=temporal_config,
        context_chunks=CONTEXT_CHUNKS,
        target_chunks=TARGET_CHUNKS,
    ).to(device).eval()
    if TEMPORAL_CHECKPOINT is not None and TEMPORAL_CHECKPOINT.is_file():
        payload = torch.load(TEMPORAL_CHECKPOINT, map_location="cpu")
        if isinstance(payload, dict):
            state = payload.get("model_state_dict") or payload.get("model") or payload.get("state_dict")
        else:
            state = payload
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"loaded temporal checkpoint: {TEMPORAL_CHECKPOINT}")
        print(f"temporal load missing={len(missing)} unexpected={len(unexpected)}")
    return model

model = build_temporal_model(DEVICE)
for p in model.event_encoder.parameters():
    assert not p.requires_grad

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"total_params={total_params:,}")
print(f"trainable_temporal_head_params={trainable_params:,}")
print(f"frozen_encoder_params={frozen_params:,}")

In [ ]:
context_header = torch.randint(0, 256, (BATCH_SIZE, CONTEXT_CHUNKS, HEADER_BYTES), dtype=torch.uint8, device=DEVICE)
context_events = torch.randint(0, 256, (BATCH_SIZE, CONTEXT_CHUNKS, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=DEVICE)

with torch.no_grad():
    output = model(context_header, context_events)

print("chunk_embeddings", tuple(output.chunk_embeddings.shape))
print("header_bit_logits", tuple(output.header_bit_logits.shape))
print("event_bit_logits", tuple(output.event_bit_logits.shape))

In [ ]:
class TemporalTensorSummaryModel(nn.Module):
    """Mirror TemporalEventPredictor but return plain tensors for torchinfo/torchview."""

    def __init__(self, model: TemporalEventPredictor) -> None:
        super().__init__()
        self.event_encoder = model.event_encoder
        self.chunk_embedding_to_temporal_width = model.chunk_embedding_to_temporal_width
        self.context_position_embedding = model.context_position_embedding
        self.temporal_encoder = model.temporal_encoder
        self.future_chunk_query_embedding = model.future_chunk_query_embedding
        self.future_chunk_decoder = model.future_chunk_decoder
        self.future_header_bit_head = model.future_header_bit_head
        self.future_event_bit_head = model.future_event_bit_head
        self.context_chunks = model.context_chunks
        self.target_chunks = model.target_chunks

    def encode_context_chunks(self, context_header_uint8: torch.Tensor, context_events_uint8: torch.Tensor) -> torch.Tensor:
        batch_size, context_chunks = context_header_uint8.shape[:2]
        flat_header = context_header_uint8.reshape(batch_size * context_chunks, HEADER_BYTES)
        flat_events = context_events_uint8.reshape(batch_size * context_chunks, EVENTS_PER_CHUNK, EVENT_BYTES)
        flat_embedding = self.event_encoder(flat_header, flat_events)
        return flat_embedding.reshape(batch_size, context_chunks, -1)

    def forward(self, context_header_uint8: torch.Tensor, context_events_uint8: torch.Tensor):
        chunk_embeddings = self.encode_context_chunks(context_header_uint8, context_events_uint8)
        temporal_tokens = self.chunk_embedding_to_temporal_width(chunk_embeddings)
        positions = torch.arange(self.context_chunks, device=temporal_tokens.device).view(1, -1)
        temporal_tokens = temporal_tokens + self.context_position_embedding(positions)
        temporal_memory = self.temporal_encoder(temporal_tokens)
        query_ids = torch.arange(self.target_chunks, device=temporal_tokens.device).view(1, -1).expand(temporal_tokens.shape[0], -1)
        future_queries = self.future_chunk_query_embedding(query_ids)
        decoded_future = self.future_chunk_decoder(future_queries, temporal_memory)
        header_logits = self.future_header_bit_head(decoded_future).view(
            decoded_future.shape[0], self.target_chunks, HEADER_BYTES, BITS_PER_BYTE
        )
        event_logits = self.future_event_bit_head(decoded_future).view(
            decoded_future.shape[0], self.target_chunks, EVENTS_PER_CHUNK, EVENT_BYTES, BITS_PER_BYTE
        )
        return header_logits, event_logits, chunk_embeddings

summary_model = TemporalTensorSummaryModel(model).eval()
try:
    from torchinfo import summary
    summary(
        summary_model,
        input_data=(context_header, context_events),
        depth=8,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=1,
    )
except Exception as exc:
    print(f"torchinfo summary failed: {exc!r}")

In [ ]:
try:
    from torchview import draw_graph
    graph = draw_graph(
        summary_model,
        input_data=(context_header, context_events),
        expand_nested=True,
        graph_name="temporal_v1_real_v20_encoder",
        save_graph=False,
    )
    graph.visual_graph
except Exception as exc:
    print(f"torchview graph failed: {exc!r}")

In [ ]:
with torch.no_grad():
    header_probs = torch.sigmoid(output.header_bit_logits).detach().cpu()
    event_probs = torch.sigmoid(output.event_bit_logits).detach().cpu()

print("header prob range", float(header_probs.min()), float(header_probs.max()))
print("event prob range", float(event_probs.min()), float(event_probs.max()))
print("first target header byte-bit probs", header_probs[0, 0, 0].tolist())